# Employee churn pipeline — notebook (EDA + clustering)

> **Portfolio note:** Production training and scoring live in `../src/` and `../scripts/`.
> Run `python scripts/train.py` for reproducible metrics and saved models.
> Clear outputs before git commit: `jupyter nbconvert --clear-output --inplace notebooks/employee_churn_pipeline.ipynb`


In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML & preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)

# Clustering
from sklearn.cluster import KMeans

# Imbalanced data
from imblearn.over_sampling import SMOTE

# Settings
plt.style.use("default")
sns.set_theme()


In [ ]:
# Load the dataset
data = pd.read_csv('HR_comma_sep.csv')
copy_data = data.copy()
# Clean column names (remove extra spaces)
copy_data.columns = copy_data.columns.str.strip()

print("Dataset shape:", copy_data.shape)
print("\nFirst few rows:")
print(copy_data.head())
print("\nDataset info:")
print(copy_data.info())
print(copy_data.describe())
print("\nColumn names:")
print(copy_data.columns.tolist())
print("\nBasic statistics:")


In [ ]:
# Checking null values

In [ ]:
copy_data.isnull().sum()


In [ ]:
# 3.1 Correlation Heatmap (Numerical Features)

In [ ]:
plt.figure(figsize=(10, 6))
corr = copy_data.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Numerical Features")
plt.show()


In [ ]:
# 3.2 Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Satisfaction Level Distribution
sns.histplot(
    copy_data["satisfaction_level"],
    bins=30,
    kde=True,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Distribution of Employee Satisfaction Level")
axes[0, 0].set_xlabel("Satisfaction Level")

# 2. Last Evaluation Distribution
sns.histplot(
    copy_data["last_evaluation"],
    bins=30,
    kde=True,
    ax=axes[0, 1]
)
axes[0, 1].set_title("Distribution of Employee Last Evaluation")
axes[0, 1].set_xlabel("Evaluation Score")

# 3. Average Monthly Hours Distribution
sns.histplot(
    copy_data["average_montly_hours"],
    bins=30,
    kde=True,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Distribution of Average Monthly Working Hours")
axes[1, 0].set_xlabel("Monthly Hours")

# 4. Project Count vs Turnover
sns.countplot(
    x="number_project",
    hue="left",
    data=copy_data,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Number of Projects vs Employee Turnover")
axes[1, 1].set_xlabel("Number of Projects")
axes[1, 1].set_ylabel("Employee Count")
axes[1, 1].legend(title="Left")

plt.tight_layout()
plt.show()


In [ ]:
# STEP 4: Clustering Employees Who Left

In [ ]:
# 4.1 Filter Employees Who Left

In [ ]:
df_left = copy_data[copy_data["left"] == 1].copy()
df_left.shape

In [ ]:
# 4.2 Select Features for Clustering

In [ ]:
X_cluster = df_left[["satisfaction_level", "last_evaluation"]]

In [ ]:
# 4.3 K-Means Clustering (k = 3)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42,  n_init=10)

df_left["cluster"] = kmeans.fit_predict(X_cluster)


In [ ]:
# 4.4 Visualize Clusters

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(
    x="satisfaction_level",
    y="last_evaluation",
    hue="cluster",
    palette="Set1",
    data=df_left
)
plt.title("Clusters of Employees Who Left")
plt.show()


In [ ]:
# STEP 5: Handle Class Imbalance with SMOTE

In [ ]:
# 5.1 Separate Features & Target

In [ ]:
X = copy_data.drop("left", axis=1)
y = copy_data["left"]


In [ ]:
# 5.2 Separate Categorical & Numerical Columns

In [ ]:
cat_cols = ["sales", "salary"]
num_cols = X.columns.difference(cat_cols)


In [ ]:
# 5.3 Encode Categorical Variables

In [ ]:
X_cat = pd.get_dummies(X[cat_cols], drop_first=True)
X_num = X[num_cols]

X_final = pd.concat([X_num, X_cat], axis=1)


In [ ]:
# STEP 6: Train-Test Split (Stratified 80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    stratify=y,
    random_state=123
)


In [ ]:
# STEP 7: Apply SMOTE (Training Data Only)

In [ ]:
smote = SMOTE(random_state=123)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

y_train_sm.value_counts()


In [ ]:
# STEP 8: Model Training with 5-Fold Cross-Validation

In [ ]:
# Helper Function for CV Evaluation

In [ ]:
def evaluate_model(model, X, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    y_pred = cross_val_predict(model, X, y, cv=skf)
    print(classification_report(y, y_pred))
    return y_pred


In [ ]:
# 8.1 Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
y_pred_lr = evaluate_model(log_reg, X_train_sm, y_train_sm)


In [ ]:
# 8.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=123)
y_pred_rf = evaluate_model(rf, X_train_sm, y_train_sm)


In [ ]:
# 8.3 Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(random_state=123)
y_pred_gb = evaluate_model(gb, X_train_sm, y_train_sm)


In [ ]:
# STEP 9: ROC Curve & AUC (Test Data)

In [ ]:
# Fit Models on SMOTE Data

In [ ]:
log_reg.fit(X_train_sm, y_train_sm)
rf.fit(X_train_sm, y_train_sm)
gb.fit(X_train_sm, y_train_sm)


In [ ]:
# Predict Probabilities

In [ ]:
y_prob_lr = log_reg.predict_proba(X_test)[:,1]
y_prob_rf = rf.predict_proba(X_test)[:,1]
y_prob_gb = gb.predict_proba(X_test)[:,1]


In [ ]:
# Plot ROC Curve

In [ ]:
plt.figure(figsize=(8,6))

for model_name, y_prob in zip(
    ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    [y_prob_lr, y_prob_rf, y_prob_gb]
):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.2f})")

plt.plot([0,1], [0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


In [ ]:
# STEP 10: Confusion Matrices

In [ ]:
models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf,
    "Gradient Boosting": gb
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n{name}")
    print(confusion_matrix(y_test, y_pred))


In [ ]:
# STEP 11: Predict Turnover Probability & Risk Zones

In [ ]:
df_test = X_test.copy()
df_test["turnover_probability"] = y_prob_gb


In [ ]:
# Assign Risk Zones

In [ ]:
def risk_zone(p):
    if p < 0.2:
        return "Safe (Green)"
    elif p < 0.6:
        return "Low Risk (Yellow)"
    elif p < 0.9:
        return "Medium Risk (Orange)"
    else:
        return "High Risk (Red)"

df_test["risk_zone"] = df_test["turnover_probability"].apply(risk_zone)
df_test["risk_zone"].value_counts()
